# Módulo 48 · DL IV: tus propias imágenes, aumento de datos y uso del modelo

Código del módulo para ejecutar en **Google Colab**, donde TensorFlow y PyTorch ya vienen instalados. Para entrenar más rápido activa la GPU en *Entorno de ejecución → Cambiar tipo de entorno de ejecución*. Las explicaciones, los ejercicios y las partes que funcionan en el navegador están en el módulo 48 de la web.

In [ ]:
# Solo si lo ejecutas en tu ordenador (en Colab ya está todo instalado):
# !pip install tensorflow opencv-python

### Keras: gatos frente a perros con aumento de datos (Colab, necesita el dataset PetImages)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

ancho = 125
# Lee las fotos de PetImages/Cat y PetImages/Dog; la etiqueta es el nombre de la carpeta
train_ds, val_ds = tf.keras.utils.image_dataset_from_directory(
    "PetImages", validation_split=0.2, subset="both", seed=42,
    color_mode="grayscale", image_size=(ancho, ancho), batch_size=32, label_mode="binary")

aumento = models.Sequential([layers.RandomFlip("horizontal"), layers.RandomRotation(0.08),
                             layers.RandomZoom(0.2), layers.RandomTranslation(0.1, 0.1)])

model = models.Sequential([
    layers.Input(shape=(ancho, ancho, 1)),
    aumento,                                   # solo actúa durante el entrenamiento
    layers.Rescaling(1 / 255),
    layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Conv2D(256, 3, activation="relu"),
    layers.Dropout(0.25), layers.Flatten(),
    layers.Dense(250, activation="relu"), layers.Dropout(0.25),
    layers.Dense(1, activation="sigmoid"),     # probabilidad de perro
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
parada = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
history = model.fit(train_ds, validation_data=val_ds, epochs=200, callbacks=[parada])
model.save("catvsdog.keras")

### Predicción en directo con la webcam (en tu ordenador: la webcam no es accesible desde Colab con OpenCV)

In [ ]:
import cv2
import numpy as np
import tensorflow as tf

model = tf.keras.models.load_model("modelo_mnist_convol.h5")

def preprocess_image(frame):
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    img = cv2.resize(img, (28, 28)).astype("float32") / 255
    img = (1 - img) ** 2              # papel: negro sobre blanco -> MNIST: blanco sobre negro, con más contraste
    return img[None, :, :, None]      # (1, 28, 28, 1): lote de una imagen con un canal

cap = cv2.VideoCapture(0)
while True:
    ok, frame = cap.read()
    if not ok:
        break
    probs = model.predict(preprocess_image(frame), verbose=0)[0]
    texto = f"{probs.argmax()} ({probs.max():.2f})" if probs.max() > 0.8 else "dudoso"
    cv2.putText(frame, texto, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)
    cv2.imshow("Webcam", frame)
    tecla = cv2.waitKey(1) & 0xFF           # UNA sola lectura de teclado por fotograma
    if tecla == ord("q"):
        break
cap.release()
cv2.destroyAllWindows()